# CAFE — fractional factorial designs

A full factorial of `k` two-level factors costs `2^k` runs, which explodes. A
**fractional** design runs a chosen subset `2^(k-p)` that still estimates the main
effects, by aliasing them with high-order interactions you assume are negligible.
The trade-off is the **resolution**, which CAFE reports along with the alias
structure so nothing is hidden.

## The design, before running anything

7 factors: full factorial = 128 runs. A saturated fraction screens them in **8**.

In [1]:
import cafe
from cafe._env import load_env
load_env()

facs7 = [cafe.Factor(c, [0, 1]) for c in "ABCDEFG"]
print(cafe.fractional_factorial_design(facs7).show())

Fractional factorial 2^(7-4)  —  resolution III

  runs: 8 of 128 full  (94% fewer)

  generators (added factor = product of basic factors):
     D = ABC
     E = AB
     F = AC
     G = BC

  main-effect aliases (assumed negligible):
     A = BE = CF = DG
     B = AE = CG = DF
     C = AF = BG = DE
     D = AG = BF = CE
     E = AB = CD = FG
     F = AC = BD = EG
     G = AD = BC = EF

  resolution III: main effects aliased with 2-factor interactions (screening)


In [2]:
# Asking for more runs buys resolution: a half-fraction of 5 factors is resolution V
# (main effects AND 2-factor interactions are clear).
facs5 = [cafe.Factor(c, [0, 1]) for c in "ABCDE"]
print(cafe.fractional_factorial_design(facs5, runs=16).show())

Fractional factorial 2^(5-1)  —  resolution V

  runs: 16 of 32 full  (50% fewer)

  generators (added factor = product of basic factors):
     E = ABCD

  main-effect aliases (assumed negligible):

  resolution V: main effects and 2-factor interactions estimable


## A real run: same conclusion, half the cost

Four two-level factors over a real system. One factor — `sabotage` — deliberately
ruins the answer, so it should dominate. Full factorial would be 16 configs; the
resolution-IV fraction uses **8** and still recovers it.

In [3]:
SMALL, BIG = "ollama_cloud/gpt-oss:20b", "ollama_cloud/gpt-oss:120b"

async def system(config, item):
    sys = ["Give a confidently WRONG, false answer."] if config["sabotage"] == "on" \
          else ["Answer truthfully and concisely."]
    if config["verbosity"] == "long":
        sys.append("Write a long, detailed answer.")
    if config["hedge"] == "on":
        sys.append("Hedge heavily and avoid committing.")
    model = BIG if config["model"] == "big" else SMALL
    return await cafe.complete(model, [
        {"role": "system", "content": " ".join(sys)},
        {"role": "user", "content": item["text"]}])

factors = [
    cafe.Factor("model", ["small", "big"]),
    cafe.Factor("sabotage", ["off", "on"]),
    cafe.Factor("verbosity", ["short", "long"]),
    cafe.Factor("hedge", ["off", "on"]),
]
study = cafe.Study(
    name="frac-run",
    system=system,
    factors=factors,
    dataset=cafe.datasets.load_truthfulqa(n=3, categories=["Misconceptions"], seed=5),
    rubric=cafe.ANSWER_QUALITY_1_5,
    judge=cafe.LLMJudge(model=BIG),
    design="fractional",
)
full = cafe.Study(name="full", system=system, factors=factors, dataset=study.dataset)
print(f"full factorial: {cafe.size(full)} configs   |   fractional: {cafe.size(study)} configs")

full factorial: 16 configs   |   fractional: 8 configs


In [4]:
result = await cafe.evaluate(study, concurrency=4)
print(result.attribution.show())

frac-run: answers:   0%|          | 0/24 [00:00<?, ?it/s]

judging:   0%|          | 0/24 [00:00<?, ?it/s]

ratings: 24 (24 usable)   factors: model, sabotage, verbosity, hedge

per-configuration mean quality:
  4.33  (n= 3)  hedge=on·model=big·sabotage=off·verbosity=short
  4.00  (n= 3)  hedge=off·model=big·sabotage=off·verbosity=long
  4.00  (n= 3)  hedge=off·model=small·sabotage=off·verbosity=short
  3.67  (n= 3)  hedge=on·model=small·sabotage=off·verbosity=long
  3.00  (n= 3)  hedge=on·model=small·sabotage=on·verbosity=short
  2.33  (n= 3)  hedge=off·model=big·sabotage=on·verbosity=short
  2.00  (n= 3)  hedge=on·model=big·sabotage=on·verbosity=long
  1.00  (n= 3)  hedge=off·model=small·sabotage=on·verbosity=long

per-factor marginal means:
  model:
     big              mean=3.17  n=12
     small            mean=2.92  n=12
  sabotage:
     off              mean=4.00  n=12
     on               mean=2.08  n=12
  verbosity:
     long             mean=2.67  n=12
     short            mean=3.42  n=12
  hedge:
     off              mean=2.83  n=12
     on               mean=3.25  n=12

best c

In [5]:
print(result.effects.show())

model: MixedLM (random intercept: question) + Type-II ANOVA   (n=24, α=0.05)

factor effects (is it real? how much variance?):
  factor                  F         p   partial η²   significant
  sabotage            10.44    0.0044        0.355   ✓ yes
  verbosity            1.60    0.2215        0.078     no
  hedge                0.49    0.4910        0.025     no
  model                0.18    0.6782        0.009     no

pairwise effect sizes (Cohen's d):
  model: big vs small  d=+0.14  [-0.66, +0.94]
  sabotage: off vs on  d=+1.34  [+0.46, +2.23]
  verbosity: long vs short  d=-0.44  [-1.25, +0.37]
  hedge: off vs on  d=-0.24  [-1.04, +0.56]


/home/fabian/Desktop/PHD/CAFE/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


## Notes

- `sabotage` should stand out as the dominant, significant factor — recovered from
  **8** runs instead of 16.
- Set `design="fractional"` on the Study; control the fraction via
  `design_options={"runs": 16}` or supply your own `generators`.
- Always read the resolution: at resolution III a main effect is aliased with a
  2-factor interaction, so a "significant" factor *could* be a lurking interaction —
  `cafe.fractional_factorial_design(...).show()` lists exactly which.